In [2]:
import pandas as pd

In [3]:
orders = pd.read_csv("../../data/processed/orders_clean.csv")
order_items = pd.read_csv("../../data/processed/order_items_clean.csv")
customers = pd.read_csv("../../data/processed/customers_clean.csv")

In [ ]:

products = pd.read_csv("../../data/processed/products_clean.csv")

print("Products:", products.shape)

Products: (32951, 9)


In [4]:
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Customers:", customers.shape)

Orders: (99441, 8)
Order Items: (112650, 7)
Customers: (99441, 5)


In [5]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"],
    errors="coerce"
)

orders["order_month"] = (
    orders["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

orders[["order_purchase_timestamp", "order_month"]].head()

,order_purchase_timestamp,order_month
0,2017-02-10 10:56:00,2017-02
1,NaT,NaN
2,2018-08-08 08:38:00,2018-08
3,NaT,NaN
4,NaT,NaN


In [6]:
sales_data = order_items.merge(
    orders[["order_id", "order_status", "order_purchase_timestamp"]],
    on="order_id",
    how="left"
)

sales_data["order_month"] = (
    sales_data["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

sales_data.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,order_status,order_purchase_timestamp,order_month
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,delivered,NaT,NaN
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,delivered,NaT,NaN
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,delivered,NaT,NaN
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,delivered,2018-08-08 10:00:00,2018-08
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,delivered,2017-04-02 13:57:00,2017-04


In [7]:
# Calculate monthly product sales using only delivered orders
monthly_sales = (
    sales_data[sales_data["order_status"] == "delivered"]
    .groupby("order_month", as_index=False)["price"]
    .sum()
    .rename(columns={"price": "total_sales"})
)

# Display the first 5 months to check the result
monthly_sales.head()

,order_month,total_sales
0,2016-03,441.98
1,2016-04,8595.89
2,2016-05,6169.77
3,2016-06,5889.96
4,2016-07,6075.35


In [8]:
# Install Plotly in the current Python environment
%pip install plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:

%pip install nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
# Import Plotly for creating an interactive chart
import plotly.express as px

# Create a line chart showing product sales by month
fig = px.line(
    monthly_sales,
    x="order_month",
    y="total_sales",
    title="Monthly Sales Trend",
    labels={
        "order_month": "Month",
        "total_sales": "Total Sales"
    }
)

# Display the chart
fig.show()

In [11]:
# Sort the monthly sales from highest to lowest
top_sales_months = monthly_sales.sort_values(
    by="total_sales",
    ascending=False
)

# Display the 10 months with the highest sales
top_sales_months.head(10)

,order_month,total_sales
28,2018-09,267052.61
26,2018-07,264791.91
23,2018-04,262268.78
30,2018-11,262149.26
24,2018-05,255495.89
22,2018-03,252510.08
25,2018-06,251949.12
27,2018-08,245241.92
21,2018-02,244592.46
29,2018-10,236662.31


In [12]:
# Sort monthly sales chronologically before calculating month-over-month change
monthly_sales = monthly_sales.sort_values("order_month").reset_index(drop=True)

# Calculate the percentage change in sales compared with the previous month
monthly_sales["sales_growth_pct"] = (
    monthly_sales["total_sales"].pct_change() * 100
)

# Round the growth percentage to 2 decimal places
monthly_sales["sales_growth_pct"] = monthly_sales["sales_growth_pct"].round(2)

# Display the monthly sales and growth information
monthly_sales.head(10)

,order_month,total_sales,sales_growth_pct
0,2016-03,441.98,NaN
1,2016-04,8595.89,1844.86
2,2016-05,6169.77,-28.22
3,2016-06,5889.96,-4.54
4,2016-07,6075.35,3.15
5,2016-08,7592.89,24.98
6,2016-09,2399.70,-68.40
7,2016-10,3159.57,31.67
8,2017-01,205462.54,6402.86
9,2017-02,185204.64,-9.86


In [13]:
# Create a line chart showing month-over-month sales growth
fig = px.line(
    monthly_sales,
    x="order_month",
    y="sales_growth_pct",
    title="Monthly Sales Growth",
    labels={
        "order_month": "Month",
        "sales_growth_pct": "Sales Growth (%)"
    }
)

# Display the chart
fig.show()

In [14]:
# Display the earliest months so we can investigate the unusually high growth
monthly_sales.head(10)

,order_month,total_sales,sales_growth_pct
0,2016-03,441.98,NaN
1,2016-04,8595.89,1844.86
2,2016-05,6169.77,-28.22
3,2016-06,5889.96,-4.54
4,2016-07,6075.35,3.15
5,2016-08,7592.89,24.98
6,2016-09,2399.70,-68.40
7,2016-10,3159.57,31.67
8,2017-01,205462.54,6402.86
9,2017-02,185204.64,-9.86


In [15]:
# Display monthly sales in chronological order
monthly_sales.head(15)

,order_month,total_sales,sales_growth_pct
0,2016-03,441.98,NaN
1,2016-04,8595.89,1844.86
2,2016-05,6169.77,-28.22
3,2016-06,5889.96,-4.54
4,2016-07,6075.35,3.15
5,2016-08,7592.89,24.98
6,2016-09,2399.70,-68.40
7,2016-10,3159.57,31.67
8,2017-01,205462.54,6402.86
9,2017-02,185204.64,-9.86


In [18]:
# Combine order items with product information
category_sales = order_items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

# Calculate total product sales for each category
category_sales = (
    category_sales
    .groupby("product_category_name", as_index=False)["price"]
    .sum()
    .rename(columns={"price": "total_sales"})
)

# Sort categories from highest sales to lowest sales
category_sales = category_sales.sort_values(
    by="total_sales",
    ascending=False
)

# Display the top 10 categories by sales
category_sales.head(10)

,product_category_name,total_sales
11,beleza_saude,1258681.34
66,relogios_presentes,1205005.68
13,cama_mesa_banho,1036988.68
32,esporte_lazer,988048.97
44,informatica_acessorios,911954.32
54,moveis_decoracao,729762.49
26,cool_stuff,635290.85
72,utilidades_domesticas,632248.66
8,automotivo,592720.11
40,ferramentas_jardim,485256.46


In [19]:
# Select the 10 product categories with the highest sales
top_categories = category_sales.head(10)

# Create a bar chart showing sales by product category
fig = px.bar(
    top_categories,
    x="product_category_name",
    y="total_sales",
    title="Top 10 Product Categories by Sales",
    labels={
        "product_category_name": "Product Category",
        "total_sales": "Total Sales"
    }
)

# Display the chart
fig.show()

In [20]:
# Load the category translation data
category_translation = pd.read_csv(
    "../../data/processed/category_translation_clean.csv"
)

# Check the first 5 translations
category_translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [21]:
# Add English category names to our category sales data
category_sales = category_sales.merge(
    category_translation[
        ["product_category_name", "product_category_name_english"]
    ],
    on="product_category_name",
    how="left"
)

# Display the first 10 categories with their English names and sales
category_sales.head(10)

,product_category_name,total_sales,product_category_name_english
0,beleza_saude,1258681.34,health_beauty
1,relogios_presentes,1205005.68,watches_gifts
2,cama_mesa_banho,1036988.68,bed_bath_table
3,esporte_lazer,988048.97,sports_leisure
4,informatica_acessorios,911954.32,computers_accessories
5,moveis_decoracao,729762.49,furniture_decor
6,cool_stuff,635290.85,cool_stuff
7,utilidades_domesticas,632248.66,housewares
8,automotivo,592720.11,auto
9,ferramentas_jardim,485256.46,garden_tools


In [22]:
# Select the 10 product categories with the highest sales
top_categories = category_sales.head(10)

# Create a bar chart using English category names
fig = px.bar(
    top_categories,
    x="product_category_name_english",
    y="total_sales",
    title="Top 10 Product Categories by Sales",
    labels={
        "product_category_name_english": "Product Category",
        "total_sales": "Total Sales"
    }
)

# Display the chart
fig.show()

In [24]:
# Combine order items with the product category names
category_orders = order_items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

# Add the English category names
category_orders = category_orders.merge(
    category_translation[
        ["product_category_name", "product_category_name_english"]
    ],
    on="product_category_name",
    how="left"
)

# Count the number of items sold in each category
category_orders = (
    category_orders
    .groupby("product_category_name_english", as_index=False)
    .size()
    .rename(columns={"size": "items_sold"})
)

# Sort categories from highest item volume to lowest
category_orders = category_orders.sort_values(
    by="items_sold",
    ascending=False
)

# Display the top 10 categories by number of items sold
category_orders.head(10)

,product_category_name_english,items_sold
7,bed_bath_table,11115
43,health_beauty,9670
65,sports_leisure,8641
39,furniture_decor,8334
15,computers_accessories,7827
49,housewares,6964
70,watches_gifts,5991
68,telephony,4545
42,garden_tools,4347
5,auto,4235


In [25]:
# Select the 10 categories with the highest number of items sold
top_item_categories = category_orders.head(10)

# Create a bar chart showing item volume by category
fig = px.bar(
    top_item_categories,
    x="product_category_name_english",
    y="items_sold",
    title="Top 10 Product Categories by Items Sold",
    labels={
        "product_category_name_english": "Product Category",
        "items_sold": "Items Sold"
    }
)

# Display the chart
fig.show()

In [26]:
# Combine order items with order information
state_sales = order_items.merge(
    orders[["order_id", "order_status", "customer_id"]],
    on="order_id",
    how="left"
)

# Add customer state information
state_sales = state_sales.merge(
    customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left"
)

# Calculate total product sales for delivered orders by state
state_sales = (
    state_sales[state_sales["order_status"] == "delivered"]
    .groupby("customer_state", as_index=False)["price"]
    .sum()
    .rename(columns={"price": "total_sales"})
)

# Sort states from highest sales to lowest
state_sales = state_sales.sort_values(
    by="total_sales",
    ascending=False
)

# Display the top 10 states by sales
state_sales.head(10)

,customer_state,total_sales
25,SP,5067633.16
18,RJ,1759651.13
10,MG,1552481.83
22,RS,728897.47
17,PR,666063.51
23,SC,507012.13
4,BA,493584.14
6,DF,296498.41
8,GO,282836.70
7,ES,268643.45


In [27]:
# Select the 10 states with the highest product sales
top_states = state_sales.head(10)

# Create a bar chart showing sales by customer state
fig = px.bar(
    top_states,
    x="customer_state",
    y="total_sales",
    title="Top 10 Customer States by Sales",
    labels={
        "customer_state": "Customer State",
        "total_sales": "Total Sales"
    }
)

# Display the chart
fig.show()

In [28]:
# Load the cleaned payment data
payments = pd.read_csv("../../data/processed/payments_clean.csv")

# Check the number of rows and columns
print("Payments:", payments.shape)

Payments: (103886, 5)


In [29]:
# Count how many payment records exist for each payment type
payment_methods = (
    payments
    .groupby("payment_type", as_index=False)
    .size()
    .rename(columns={"size": "payment_count"})
)

# Sort payment methods from most common to least common
payment_methods = payment_methods.sort_values(
    by="payment_count",
    ascending=False
)

# Display the payment method counts
payment_methods

,payment_type,payment_count
1,credit_card,76795
0,boleto,19784
4,voucher,5775
2,debit_card,1529
3,not_defined,3


In [ ]:
# Create a bar chart showing the number of payment records by payment type
fig = px.bar(
    payment_methods,
    x="payment_type",
    y="payment_count",
    title="Payment Method Distribution",
    labels={
        "payment_type": "Payment Method",
        "payment_count": "Payment Records"
    }
)

# Display the chart
fig.show()

#Remember: this chart counts payment records, not unique orders. An order can have multiple payment records.

In [31]:
# Calculate the total payment value for each payment method
payment_value = (
    payments
    .groupby("payment_type", as_index=False)["payment_value"]
    .sum()
    .rename(columns={"payment_value": "total_payment_value"})
)

# Sort payment methods from highest payment value to lowest
payment_value = payment_value.sort_values(
    by="total_payment_value",
    ascending=False
)

# Display the payment value by payment method
payment_value

,payment_type,total_payment_value
1,credit_card,12542084.19
0,boleto,2869361.27
4,voucher,379436.87
2,debit_card,217989.79
3,not_defined,0.00


In [32]:
# Create a bar chart showing total payment value by payment method
fig = px.bar(
    payment_value,
    x="payment_type",
    y="total_payment_value",
    title="Total Payment Value by Payment Method",
    labels={
        "payment_type": "Payment Method",
        "total_payment_value": "Total Payment Value"
    }
)

# Display the chart
fig.show()

In [33]:
# Calculate the average payment value for each payment method
average_payment = (
    payments
    .groupby("payment_type", as_index=False)["payment_value"]
    .mean()
    .rename(columns={"payment_value": "average_payment_value"})
)

# Round the average values to 2 decimal places
average_payment["average_payment_value"] = (
    average_payment["average_payment_value"].round(2)
)

# Sort payment methods from highest average value to lowest
average_payment = average_payment.sort_values(
    by="average_payment_value",
    ascending=False
)

# Display the average payment value by payment method
average_payment

,payment_type,average_payment_value
1,credit_card,163.32
0,boleto,145.03
2,debit_card,142.57
4,voucher,65.70
3,not_defined,0.00


In [35]:
# Check a few purchase and delivery dates to find the date problem
orders[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "delivery_days"
    ]
].head(20)

,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-02-10 10:56:00,2017-10-10 21:25:00,242.436806
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,NaT,2018-07-08 15:27:00,NaN
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-08 08:38:00,NaT,NaN
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,NaT,2017-02-12 00:28:00,NaN
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,NaT,NaT,NaN
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,2017-09-07 21:57:00,NaT,NaN
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,2017-11-04 12:22:00,NaT,NaN
7,6514b8ad8028c9f2cc2374ded245783f,delivered,NaT,NaT,NaN
8,76c6e866289321a7c93b82b54852dc33,delivered,NaT,2017-02-02 14:08:00,NaN
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,NaT,NaT,NaN


In [36]:
# Find delivered orders where the delivery date is earlier than the purchase date
invalid_delivery_dates = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"])
]

# Display how many invalid records were found
print("Invalid delivery dates:", len(invalid_delivery_dates))

# Display some examples
invalid_delivery_dates[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "delivery_days"
    ]
].head(10)

Invalid delivery dates: 1653


,order_id,order_purchase_timestamp,order_delivered_customer_date,delivery_days
57,66e4624ae69e7dc89bd50222b59f581f,2018-09-03 14:50:00,2018-03-04 13:28:00,-183.056944
251,bb4f0c21ae6014e40ef007ff6fbcbeb9,2017-12-02 18:48:00,2017-01-03 14:09:00,-333.193750
273,6bb1e842418aac0c9c820afd3bb63a2d,2018-11-02 14:13:00,2018-05-03 22:04:00,-182.672917
498,0fc34e566d3273325c404c113f4a8b2a,2017-02-11 14:51:00,2017-01-12 18:06:00,-29.864583
612,d396adb7e51ca1bcb4a84cb786fee813,2017-03-07 16:42:00,2017-01-08 21:10:00,-57.813889
721,7b2f3756c235e93b2ad6330c54805401,2018-12-03 16:54:00,2018-09-04 22:08:00,-89.781944
738,d8e2d2a26010eb732a91d8b6952e2c9e,2018-11-01 12:20:00,2018-06-02 22:33:00,-151.574306
753,1a4cc55b483d875e3b8bbd64a1b31bba,2018-12-04 14:05:00,2018-09-05 15:08:00,-89.956250
876,1ac605c3c8e2ee53279c7b0bd52c6aec,2018-08-03 09:26:00,2018-06-04 17:45:00,-59.653472
973,4f32c93f66aadbd682294f5cedde7c18,2018-10-03 16:15:00,2018-03-04 19:52:00,-212.849306


In [37]:
# Reload the original orders data so we can correctly parse the date columns
orders = pd.read_csv("../../data/raw/olist_orders_dataset.csv")

# List all columns that contain dates
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Convert each date column using the exact format used by the dataset
for column in date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce"
    )

# Check the corrected dates for the first order
orders[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date"
    ]
].head()

,order_id,order_purchase_timestamp,order_delivered_customer_date
0,e481f51cbdc54678b7cc49136f2d6af7,NaT,NaT
1,53cdb2fc8bc7dce0b6741e2150273451,NaT,NaT
2,47770eb9100c2d0c44946d9cf07ec65d,NaT,NaT
3,949d5b44dbf5de918fe9c16f97b45f8a,NaT,NaT
4,ad21c59c0840e6cb83a9ceb5573f8159,NaT,NaT


In [39]:
# Check a specific order that previously showed an incorrect delivery time
orders[
    orders["order_id"] == "e481f51cbdc54678b7cc49136f2d6af7"
][
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date"
    ]
]

,order_id,order_purchase_timestamp,order_delivered_customer_date
0,e481f51cbdc54678b7cc49136f2d6af7,NaT,NaT


In [40]:
# Reload the raw orders data without changing the date columns
orders_raw = pd.read_csv("../../data/raw/olist_orders_dataset.csv")

# Display the original date values for one known order
orders_raw[
    orders_raw["order_id"] == "e481f51cbdc54678b7cc49136f2d6af7"
][
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date"
    ]
]

,order_id,order_purchase_timestamp,order_delivered_customer_date
0,e481f51cbdc54678b7cc49136f2d6af7,02-10-2017 10:56,10-10-2017 21:25


In [41]:
# Display several original purchase dates exactly as stored in the CSV
orders_raw[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date"
    ]
].head(10)

,order_id,order_purchase_timestamp,order_delivered_customer_date
0,e481f51cbdc54678b7cc49136f2d6af7,02-10-2017 10:56,10-10-2017 21:25
1,53cdb2fc8bc7dce0b6741e2150273451,24-07-2018 20:41,07-08-2018 15:27
2,47770eb9100c2d0c44946d9cf07ec65d,08-08-2018 08:38,17-08-2018 18:06
3,949d5b44dbf5de918fe9c16f97b45f8a,18-11-2017 19:28,02-12-2017 00:28
4,ad21c59c0840e6cb83a9ceb5573f8159,13-02-2018 21:18,16-02-2018 18:17
5,a4591c265e18cb1dcee52889e2d8acc3,09-07-2017 21:57,26-07-2017 10:57
6,136cce7faa42fdb2cefd53fdc79a6098,11-04-2017 12:22,NaN
7,6514b8ad8028c9f2cc2374ded245783f,16-05-2017 13:10,26-05-2017 12:55
8,76c6e866289321a7c93b82b54852dc33,23-01-2017 18:29,02-02-2017 14:08
9,e69bfb5eb88e0ed6a785585b27e16dbf,29-07-2017 11:55,16-08-2017 17:14


In [42]:
# Define the date columns that need to be converted
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Convert each date column using the actual DD-MM-YYYY HH:MM format
for column in date_columns:
    orders_raw[column] = pd.to_datetime(
        orders_raw[column],
        format="%d-%m-%Y %H:%M",
        errors="coerce"
    )

# Check the first few converted dates
orders_raw[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date"
    ]
].head(10)

,order_id,order_purchase_timestamp,order_delivered_customer_date
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:00,2017-10-10 21:25:00
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:00,2018-08-07 15:27:00
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:00,2018-08-17 18:06:00
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:00,2017-12-02 00:28:00
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:00,2018-02-16 18:17:00
5,a4591c265e18cb1dcee52889e2d8acc3,2017-07-09 21:57:00,2017-07-26 10:57:00
6,136cce7faa42fdb2cefd53fdc79a6098,2017-04-11 12:22:00,NaT
7,6514b8ad8028c9f2cc2374ded245783f,2017-05-16 13:10:00,2017-05-26 12:55:00
8,76c6e866289321a7c93b82b54852dc33,2017-01-23 18:29:00,2017-02-02 14:08:00
9,e69bfb5eb88e0ed6a785585b27e16dbf,2017-07-29 11:55:00,2017-08-16 17:14:00


In [43]:
# Calculate delivery time in days for each order
orders_raw["delivery_days"] = (
    orders_raw["order_delivered_customer_date"]
    - orders_raw["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

# Keep only delivered orders with both dates available
delivery_data = orders_raw[
    (orders_raw["order_status"] == "delivered") &
    (orders_raw["delivery_days"].notna())
].copy()

# Display a few delivery times to verify the calculation
delivery_data[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "delivery_days"
    ]
].head(10)

,order_id,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:00,2017-10-10 21:25:00,8.436806
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:00,2018-08-07 15:27:00,13.781944
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:00,2018-08-17 18:06:00,9.394444
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:00,2017-12-02 00:28:00,13.208333
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:00,2018-02-16 18:17:00,2.874306
5,a4591c265e18cb1dcee52889e2d8acc3,2017-07-09 21:57:00,2017-07-26 10:57:00,16.541667
7,6514b8ad8028c9f2cc2374ded245783f,2017-05-16 13:10:00,2017-05-26 12:55:00,9.989583
8,76c6e866289321a7c93b82b54852dc33,2017-01-23 18:29:00,2017-02-02 14:08:00,9.818750
9,e69bfb5eb88e0ed6a785585b27e16dbf,2017-07-29 11:55:00,2017-08-16 17:14:00,18.221528
10,e6ce16cb79ec1d90b1da9085a6118aeb,2017-05-16 19:41:00,2017-05-29 11:18:00,12.650694


In [44]:
# Calculate delivery time in days using the correctly parsed dates
orders_raw["delivery_days"] = (
    orders_raw["order_delivered_customer_date"]
    - orders_raw["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

# Find delivered orders with a negative delivery time
invalid_delivery = orders_raw[
    (orders_raw["order_status"] == "delivered") &
    (orders_raw["delivery_days"] < 0)
]

# Display the number of invalid delivery records
print("Negative delivery times:", len(invalid_delivery))

Negative delivery times: 0


In [45]:
# Create the cleaned orders dataset from the correctly parsed raw data
orders_clean = orders_raw.drop(columns=["delivery_days"]).copy()

# Standardize the column names
orders_clean.columns = [
    column.lower().strip().replace(" ", "_")
    for column in orders_clean.columns
]

# Save the corrected orders data to the processed folder
orders_clean.to_csv(
    "../../data/processed/orders_clean.csv",
    index=False
)

# Confirm the processed file was saved
print("Corrected orders_clean.csv saved successfully.")

Corrected orders_clean.csv saved successfully.


In [47]:
# Import SQLite so we can update our project database
import sqlite3

In [48]:
# Connect to the SQLite database
conn = sqlite3.connect("../../data/processed/ecommerce.db")

# Replace the orders table with the corrected orders data
orders_clean.to_sql(
    "orders",
    conn,
    if_exists="replace",
    index=False
)

# Confirm the orders table row count
print(
    "Orders in database:",
    pd.read_sql_query(
        "SELECT COUNT(*) AS row_count FROM orders",
        conn
    )
)

# Close the database connection
conn.close()

Orders in database:    row_count
0      99441


In [51]:
# Reload the corrected orders dataset
orders = pd.read_csv("../../data/processed/orders_clean.csv")

# Convert the saved date columns back into datetime format
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Convert each date column to datetime
for column in date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        errors="coerce"
    )

# Check the dataset size
print("Orders shape:", orders.shape)

# Check the order status values
print("\nOrder status counts:")
print(orders["order_status"].value_counts())

# Check the first few dates
print("\nFirst 5 orders:")
print(
    orders[
        [
            "order_id",
            "order_purchase_timestamp",
            "order_delivered_customer_date"
        ]
    ].head()
)

Orders shape: (99441, 8)

Order status counts:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

First 5 orders:
                           order_id order_purchase_timestamp  \
0  e481f51cbdc54678b7cc49136f2d6af7      2017-10-02 10:56:00   
1  53cdb2fc8bc7dce0b6741e2150273451      2018-07-24 20:41:00   
2  47770eb9100c2d0c44946d9cf07ec65d      2018-08-08 08:38:00   
3  949d5b44dbf5de918fe9c16f97b45f8a      2017-11-18 19:28:00   
4  ad21c59c0840e6cb83a9ceb5573f8159      2018-02-13 21:18:00   

  order_delivered_customer_date  
0           2017-10-10 21:25:00  
1           2018-08-07 15:27:00  
2           2018-08-17 18:06:00  
3           2017-12-02 00:28:00  
4           2018-02-16 18:17:00  


In [52]:
# Calculate delivery time in days
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

# Keep only delivered orders with a valid delivery time
delivery_data = orders[
    (orders["order_status"] == "delivered") &
    (orders["delivery_days"].notna())
].copy()

# Create a month column using the purchase date
delivery_data["order_month"] = (
    delivery_data["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

# Calculate average delivery time for each month
monthly_delivery = (
    delivery_data
    .groupby("order_month", as_index=False)["delivery_days"]
    .mean()
    .rename(columns={"delivery_days": "average_delivery_days"})
)

# Round the average delivery time to 2 decimal places
monthly_delivery["average_delivery_days"] = (
    monthly_delivery["average_delivery_days"].round(2)
)

# Display the first 10 months
monthly_delivery.head(10)

,order_month,average_delivery_days
0,2016-09,54.81
1,2016-10,19.60
2,2016-12,4.69
3,2017-01,12.65
4,2017-02,13.17
5,2017-03,12.95
6,2017-04,14.92
7,2017-05,11.32
8,2017-06,12.01
9,2017-07,11.59


In [53]:
# Create a line chart showing average delivery time by month
fig = px.line(
    monthly_delivery,
    x="order_month",
    y="average_delivery_days",
    title="Average Delivery Time by Month",
    labels={
        "order_month": "Month",
        "average_delivery_days": "Average Delivery Time (Days)"
    }
)

# Add markers to make each month's value easier to see
fig.update_traces(
    mode="lines+markers"
)

# Display the interactive chart
fig.show()

In [54]:
# Combine order items with the corrected order information
sales_data = order_items.merge(
    orders[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp"
        ]
    ],
    on="order_id",
    how="left"
)

# Create a month column from the corrected purchase date
sales_data["order_month"] = (
    sales_data["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

# Calculate monthly product sales using delivered orders only
monthly_sales = (
    sales_data[
        sales_data["order_status"] == "delivered"
    ]
    .groupby("order_month", as_index=False)["price"]
    .sum()
    .rename(columns={"price": "total_sales"})
)

# Sort the months chronologically
monthly_sales = monthly_sales.sort_values(
    by="order_month"
).reset_index(drop=True)

# Display the first 10 months
monthly_sales.head(10)

,order_month,total_sales
0,2016-09,134.97
1,2016-10,40325.11
2,2016-12,10.90
3,2017-01,111798.36
4,2017-02,234223.40
5,2017-03,359198.85
6,2017-04,340669.68
7,2017-05,489338.25
8,2017-06,421923.37
9,2017-07,481604.52


In [55]:
# Create a line chart showing the corrected monthly sales trend
fig = px.line(
    monthly_sales,
    x="order_month",
    y="total_sales",
    title="Monthly Sales Trend",
    labels={
        "order_month": "Month",
        "total_sales": "Total Sales"
    }
)

# Add markers to make monthly values easier to identify
fig.update_traces(
    mode="lines+markers"
)

# Display the interactive chart
fig.show()

In [56]:
# Calculate the percentage change in sales compared with the previous month
monthly_sales["sales_growth_pct"] = (
    monthly_sales["total_sales"].pct_change() * 100
)

# Round the growth percentage to 2 decimal places
monthly_sales["sales_growth_pct"] = (
    monthly_sales["sales_growth_pct"].round(2)
)

# Display monthly sales and their growth rate
monthly_sales.head(15)

,order_month,total_sales,sales_growth_pct
0,2016-09,134.97,NaN
1,2016-10,40325.11,29777.09
2,2016-12,10.90,-99.97
3,2017-01,111798.36,1025573.03
4,2017-02,234223.40,109.51
5,2017-03,359198.85,53.36
6,2017-04,340669.68,-5.16
7,2017-05,489338.25,43.64
8,2017-06,421923.37,-13.78
9,2017-07,481604.52,14.15


In [57]:
# Keep the months from February 2017 onward for a more meaningful growth chart
growth_chart_data = monthly_sales[
    monthly_sales["order_month"] >= "2017-02"
].copy()

# Create a line chart showing meaningful month-over-month sales growth
fig = px.line(
    growth_chart_data,
    x="order_month",
    y="sales_growth_pct",
    title="Monthly Sales Growth",
    labels={
        "order_month": "Month",
        "sales_growth_pct": "Sales Growth (%)"
    }
)

# Add markers to make monthly changes easier to identify
fig.update_traces(
    mode="lines+markers"
)

# Display the interactive chart
fig.show()


In [58]:
# Load the cleaned review data
reviews = pd.read_csv(
    "../../data/processed/reviews_clean.csv"
)

# Check the number of reviews
print("Reviews:", reviews.shape)

# Display the first 5 reviews
reviews.head()

Reviews: (99224, 7)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [59]:
# Count the number of reviews for each score
review_distribution = (
    reviews
    .groupby("review_score", as_index=False)
    .size()
    .rename(columns={"size": "review_count"})
)

# Sort the scores from lowest to highest
review_distribution = review_distribution.sort_values(
    by="review_score"
)

# Display the review score distribution
review_distribution

,review_score,review_count
0,1,11424
1,2,3151
2,3,8179
3,4,19142
4,5,57328


In [60]:
# Create a bar chart showing the distribution of review scores
fig = px.bar(
    review_distribution,
    x="review_score",
    y="review_count",
    title="Customer Review Score Distribution",
    labels={
        "review_score": "Review Score",
        "review_count": "Number of Reviews"
    }
)

# Display the interactive chart
fig.show()

In [61]:
# Combine review scores with order delivery information
delivery_reviews = reviews.merge(
    orders[
        [
            "order_id",
            "order_status",
            "delivery_days"
        ]
    ],
    on="order_id",
    how="inner"
)

# Keep only delivered orders with a valid review score and delivery time
delivery_reviews = delivery_reviews[
    (delivery_reviews["order_status"] == "delivered") &
    (delivery_reviews["review_score"].notna()) &
    (delivery_reviews["delivery_days"].notna())
].copy()

# Calculate the average delivery time for each review score
delivery_by_review = (
    delivery_reviews
    .groupby("review_score", as_index=False)["delivery_days"]
    .mean()
    .rename(columns={"delivery_days": "average_delivery_days"})
)

# Round the average delivery time
delivery_by_review["average_delivery_days"] = (
    delivery_by_review["average_delivery_days"].round(2)
)

# Display the result
delivery_by_review

,review_score,average_delivery_days
0,1,21.31
1,2,16.66
2,3,14.26
3,4,12.31
4,5,10.69


In [62]:
# Create a bar chart showing average delivery time for each review score
fig = px.bar(
    delivery_by_review,
    x="review_score",
    y="average_delivery_days",
    title="Average Delivery Time by Review Score",
    labels={
        "review_score": "Review Score",
        "average_delivery_days": "Average Delivery Time (Days)"
    }
)

# Display the interactive chart
fig.show()

In [63]:
# Calculate the correlation between delivery time and review score
delivery_review_correlation = (
    delivery_reviews["delivery_days"]
    .corr(delivery_reviews["review_score"])
)

# Display the correlation value
print(
    "Correlation between delivery time and review score:",
    round(delivery_review_correlation, 3)
)

Correlation between delivery time and review score: -0.334


In [64]:
# Sort months by average delivery time from highest to lowest
worst_delivery_months = monthly_delivery.sort_values(
    by="average_delivery_days",
    ascending=False
)

# Display the 10 months with the longest average delivery time
worst_delivery_months.head(10)

,order_month,average_delivery_days
0,2016-09,54.81
1,2016-10,19.60
16,2018-02,16.95
17,2018-03,16.30
14,2017-12,15.39
13,2017-11,15.16
6,2017-04,14.92
15,2018-01,14.08
4,2017-02,13.17
5,2017-03,12.95


In [ ]:
# Combine order items with order status information
seller_sales = order_items.merge(
    orders[
        [
            "order_id",
            "order_status"
        ]
    ],
    on="order_id",
    how="left"
)

# Keep only items from delivered orders
seller_sales = seller_sales[
    seller_sales["order_status"] == "delivered"
].copy()

# Calculate total product sales for each seller
seller_sales = (
    seller_sales
    .groupby("seller_id", as_index=False)["price"]
    .sum()
    .rename(columns={"price": "total_sales"})
)

# Sort sellers from highest sales to lowest
seller_sales = seller_sales.sort_values(
    by="total_sales",
    ascending=False
)

# Display the 10 highest-selling sellers
seller_sales.head(10)



####This can eventually support business questions such as:
##"Which sellers contribute the most sales?"
#and later:
#"Are our highest-selling sellers also delivering on time?"

,seller_id,total_sales
834,4869f7a5dfa277a7dca6462dcf3b52b2,226987.93
982,53243585a1d6dc2643021fd1853d8905,217940.44
858,4a3ca9315b744ce9f8e9374361493884,196882.12
2903,fa1c13f2614d7b5c4749cbc52fecda94,190917.14
1480,7c67e1448b00f6e969d365cea6b010ab,186570.05
1504,7e93a43ef30c4f03f38b393420bc753a,165981.49
2543,da8622b14eb17ae2831f4ac5b9dab84a,159816.87
1450,7a67c85e85bb2ce8582c35f2203ad736,139658.69
188,1025f0e2d44d7041d6cf58b6550e0bfa,138208.56
1758,955fee9216a65b617aa5c0531780ce60,131836.71


In [66]:
# Sort sellers by total sales from highest to lowest
top_sellers = seller_sales.head(10)

# Display the top 10 sellers
top_sellers

,seller_id,total_sales
834,4869f7a5dfa277a7dca6462dcf3b52b2,226987.93
982,53243585a1d6dc2643021fd1853d8905,217940.44
858,4a3ca9315b744ce9f8e9374361493884,196882.12
2903,fa1c13f2614d7b5c4749cbc52fecda94,190917.14
1480,7c67e1448b00f6e969d365cea6b010ab,186570.05
1504,7e93a43ef30c4f03f38b393420bc753a,165981.49
2543,da8622b14eb17ae2831f4ac5b9dab84a,159816.87
1450,7a67c85e85bb2ce8582c35f2203ad736,139658.69
188,1025f0e2d44d7041d6cf58b6550e0bfa,138208.56
1758,955fee9216a65b617aa5c0531780ce60,131836.71


In [67]:
# Create a bar chart showing the top 10 sellers by sales
fig = px.bar(
    top_sellers,
    x="seller_id",
    y="total_sales",
    title="Top 10 Sellers by Product Sales",
    labels={
        "seller_id": "Seller",
        "total_sales": "Total Sales"
    }
)

# Display the interactive chart
fig.show()

In [ ]:
# Calculate total sales across all sellers
total_seller_sales = seller_sales["total_sales"].sum()

# Calculate sales generated by the top 10 sellers
top_10_seller_sales = seller_sales.head(10)["total_sales"].sum()

# Calculate the percentage of sales contributed by the top 10 sellers
top_10_sales_share = (
    top_10_seller_sales / total_seller_sales
) * 100

# Display the result
print(
    "Top 10 sellers' sales share:",
    round(top_10_sales_share, 2),
    "%"
)

#How dependent is the marketplace on its biggest sellers?"
#\If the percentage is high, the business might have seller concentration risk — losing a few major sellers could have a noticeable effect on sales

Top 10 sellers' sales share: 13.27 %


In [69]:
# Combine order items with order delivery information
seller_delivery = order_items.merge(
    orders[
        [
            "order_id",
            "order_status",
            "delivery_days"
        ]
    ],
    on="order_id",
    how="left"
)

# Keep only delivered orders with valid delivery times
seller_delivery = seller_delivery[
    (seller_delivery["order_status"] == "delivered") &
    (seller_delivery["delivery_days"].notna())
].copy()

# Calculate average delivery time for each seller
seller_delivery = (
    seller_delivery
    .groupby("seller_id", as_index=False)["delivery_days"]
    .mean()
    .rename(columns={"delivery_days": "average_delivery_days"})
)

# Round delivery time to 2 decimal places
seller_delivery["average_delivery_days"] = (
    seller_delivery["average_delivery_days"].round(2)
)

# Display sellers with their average delivery time
seller_delivery.head(10)

,seller_id,average_delivery_days
0,0015a82c2db000af6aaaf3ae2ecb0532,10.79
1,001cca7ae9ae17fb1caed9dfb1094831,13.10
2,002100f778ceb8431b7a1020ff7ab48f,16.19
3,003554e2dce176b5555353e4f3555ac8,4.65
4,004c9cd9d87a3c30c522c48c4fc07416,14.43
5,00720abe85ba0859807595bbf045a33b,8.53
6,00ab3eff1b5192e5f1a63bcecfee11c8,9.05
7,00d8b143d12632bad99c0ad66ad52825,7.23
8,00ee68308b45bc5e2660cd833c3f81cc,9.17
9,00fc707aaaad2d31347cf883cd2dfe10,15.77


In [ ]:
# Combine seller sales with average delivery performance
seller_performance = seller_sales.merge(
    seller_delivery,
    on="seller_id",
    how="inner"
)

# Sort sellers by total sales from highest to lowest
seller_performance = seller_performance.sort_values(
    by="total_sales",
    ascending=False
)

# Display the top 10 sellers with their sales and delivery performance
seller_performance.head(10)
#This is much more useful than looking at sales alone because we can start asking:

#"Are our biggest sellers also delivering efficiently?"

,seller_id,total_sales,average_delivery_days
0,4869f7a5dfa277a7dca6462dcf3b52b2,226987.93,15.01
1,53243585a1d6dc2643021fd1853d8905,217940.44,13.37
2,4a3ca9315b744ce9f8e9374361493884,196882.12,14.42
3,fa1c13f2614d7b5c4749cbc52fecda94,190917.14,13.34
4,7c67e1448b00f6e969d365cea6b010ab,186570.05,22.39
5,7e93a43ef30c4f03f38b393420bc753a,165981.49,11.34
6,da8622b14eb17ae2831f4ac5b9dab84a,159816.87,11.17
7,7a67c85e85bb2ce8582c35f2203ad736,139658.69,11.17
8,1025f0e2d44d7041d6cf58b6550e0bfa,138208.56,12.05
9,955fee9216a65b617aa5c0531780ce60,131836.71,10.74


In [71]:
# Select the 10 highest-selling sellers
top_seller_performance = seller_performance.head(10).copy()

# Sort the top sellers by delivery time from slowest to fastest
top_seller_performance = top_seller_performance.sort_values(
    by="average_delivery_days",
    ascending=False
)

# Display the top sellers and their delivery performance
top_seller_performance

,seller_id,total_sales,average_delivery_days
4,7c67e1448b00f6e969d365cea6b010ab,186570.05,22.39
0,4869f7a5dfa277a7dca6462dcf3b52b2,226987.93,15.01
2,4a3ca9315b744ce9f8e9374361493884,196882.12,14.42
1,53243585a1d6dc2643021fd1853d8905,217940.44,13.37
3,fa1c13f2614d7b5c4749cbc52fecda94,190917.14,13.34
8,1025f0e2d44d7041d6cf58b6550e0bfa,138208.56,12.05
5,7e93a43ef30c4f03f38b393420bc753a,165981.49,11.34
6,da8622b14eb17ae2831f4ac5b9dab84a,159816.87,11.17
7,7a67c85e85bb2ce8582c35f2203ad736,139658.69,11.17
9,955fee9216a65b617aa5c0531780ce60,131836.71,10.74


In [72]:
# Create a bar chart showing average delivery time for the top-selling sellers
fig = px.bar(
    top_seller_performance,
    x="seller_id",
    y="average_delivery_days",
    title="Average Delivery Time of Top 10 Sellers",
    labels={
        "seller_id": "Seller",
        "average_delivery_days": "Average Delivery Time (Days)"
    }
)

# Display the interactive chart
fig.show()

In [74]:
#Customer Repeat Purchase Analysis
# Combine orders with customer information
customer_orders = orders.merge(
    customers[
        [
            "customer_id",
            "customer_unique_id"
        ]
    ],
    on="customer_id",
    how="left"
)

# Count how many orders each unique customer placed
orders_per_customer = (
    customer_orders
    .groupby("customer_unique_id", as_index=False)
    .size()
    .rename(columns={"size": "order_count"})
)

# Count how many customers fall into each order-count group
order_frequency = (
    orders_per_customer
    .groupby("order_count", as_index=False)
    .size()
    .rename(columns={"size": "customer_count"})
)

# Sort by number of orders
order_frequency = order_frequency.sort_values(
    by="order_count"
)

# Display the first 10 order-frequency groups
order_frequency.head(10)

,order_count,customer_count
0,1,93099
1,2,2745
2,3,203
3,4,30
4,5,8
5,6,6
6,7,3
7,9,1
8,17,1


In [75]:
# Count customers who placed more than one order
repeat_customers = (
    orders_per_customer[
        orders_per_customer["order_count"] > 1
    ].shape[0]
)

# Count all unique customers
total_unique_customers = orders_per_customer.shape[0]

# Calculate the repeat customer rate
repeat_customer_rate = (
    repeat_customers / total_unique_customers
) * 100

# Display the results
print("Total unique customers:", total_unique_customers)
print("Repeat customers:", repeat_customers)
print(
    "Repeat customer rate:",
    round(repeat_customer_rate, 2),
    "%"
)

Total unique customers: 96096
Repeat customers: 2997
Repeat customer rate: 3.12 %


In [76]:
# Create a bar chart showing how many customers placed each number of orders
fig = px.bar(
    order_frequency,
    x="order_count",
    y="customer_count",
    title="Customer Order Frequency",
    labels={
        "order_count": "Number of Orders",
        "customer_count": "Number of Customers"
    }
)

# Display the interactive chart
fig.show()

In [78]:
#Top Products by Sales
# Combine order items with order status
product_sales = order_items.merge(
    orders[
        [
            "order_id",
            "order_status"
        ]
    ],
    on="order_id",
    how="left"
)

# Keep only products from delivered orders
product_sales = product_sales[
    product_sales["order_status"] == "delivered"
].copy()

# Calculate total sales for each product
product_sales = (
    product_sales
    .groupby("product_id", as_index=False)["price"]
    .sum()
    .rename(columns={"price": "total_sales"})
)

# Sort products from highest sales to lowest
product_sales = product_sales.sort_values(
    by="total_sales",
    ascending=False
)

# Display the top 10 products by sales
product_sales.head(10)

,product_id,total_sales
23546,bb50f2e236e5eea0100680137654686c,63560.00
13749,6cdd53843498f92890544667809f1595,53652.30
26997,d6160fb7873f184099d9bc95e30376af,45949.35
26436,d1c427060a0f73f6b889a5c7c61f2ac4,45620.56
19290,99a4788cb24856965c36a24e339b6058,42049.66
7881,3dd2a17168ec895c781a9191c1e95ad7,40782.80
4892,25c38557cf793876c5abdd5931f922db,38907.32
12074,5f504b3a1c75b73d6151be81eb05bdc9,37733.90
10616,53b36df67ebb7c41585e8d54d6772e08,37454.63
21617,aca2eb7d00ea1a7b8ebd4e68314663af,37104.30


In [79]:
# Calculate total product value for each order
order_value = (
    order_items
    .groupby("order_id", as_index=False)["price"]
    .sum()
    .rename(columns={"price": "order_product_value"})
)

# Add order status to each order value
order_value = order_value.merge(
    orders[
        [
            "order_id",
            "order_status"
        ]
    ],
    on="order_id",
    how="left"
)

# Keep only delivered orders
delivered_order_value = order_value[
    order_value["order_status"] == "delivered"
].copy()

# Calculate the average product value per delivered order
average_order_value = delivered_order_value[
    "order_product_value"
].mean()

# Display the result
print(
    "Average product value per delivered order:",
    round(average_order_value, 2)
)

Average product value per delivered order: 137.04


In [80]:
# Calculate the distribution of delivered order values
order_value_summary = delivered_order_value["order_product_value"].describe()

# Display the order value statistics
order_value_summary

count    96478.000000
mean       137.041586
std        209.045198
min          0.850000
25%         45.900000
50%         86.575000
75%        149.900000
max      13440.000000
Name: order_product_value, dtype: float64